In [ ]:
extending/growing domain during the quasi-Newton stage.

import time
from collections.abc import Callable
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import optax
import optimistix as optx


jax.config.update("jax_enable_x64", True)

# Domain
X0, Y0 = -1.0, -1.0
LX, LY = 2.0, 2.0

# Helmholtz parameters
A1 = 1.0
A2 = 4.0
K = 10.0

# Network
SEED = 1234
UNITS = 30
KMAX = 10

# Collocation points
N_F = 10000

# ADAM / RAdam warmup
ADAM_EPOCHS = 200
ADAM_LR = 1e-4
ADAM_PRINT_EVERY = 100

# Staged quasi-Newton training on growing domains
QN_METHOD = "SSBFGSTrustRegion"
QN_STAGES = [
    (0.2, 10000),
    (0.4, 15000),
    (0.7, 20000),
    (1.0, 40000),
]
RTOL = 1e-16
ATOL = 1e-16


def generate_inputs(key: jr.PRNGKey, n_points: int) -> jnp.ndarray:
    """Sample uniformly from the full domain [-1, 1] x [-1, 1]."""
    xy_unit = jr.uniform(key, shape=(n_points, 2))
    x = X0 + LX * xy_unit[:, 0:1]
    y = Y0 + LY * xy_unit[:, 1:2]
    return jnp.hstack([x, y])


def generate_inputs_growing(key: jr.PRNGKey, n_points: int, limit: float) -> jnp.ndarray:
    """Sample uniformly from [-limit, limit] x [-limit, limit]."""
    xy_unit = jr.uniform(key, shape=(n_points, 2))
    return (2.0 * xy_unit - 1.0) * limit



class PeriodicC0Layer(eqx.Module):
    kmax: int
    x0: float
    y0: float
    lx: float
    ly: float

    def __init__(
        self,
        kmax: int = 10,
        x0: float = -1.0,
        y0: float = -1.0,
        lx: float = 2.0,
        ly: float = 2.0,):
        
        self.kmax = kmax
        self.x0 = x0
        self.y0 = y0
        self.lx = lx
        self.ly = ly

    def __call__(self, inputs: jnp.ndarray) -> jnp.ndarray:
        x = inputs[..., 0:1]
        y = inputs[..., 1:2]

        x_shift = x - self.x0
        y_shift = y - self.y0

        ks = jnp.arange(1, self.kmax + 1, dtype=inputs.dtype)

        x_periodic = 2.0 * jnp.pi * x_shift * ks / self.lx
        y_periodic = 2.0 * jnp.pi * y_shift * ks / self.ly

        return jnp.concatenate(
            [
                jnp.cos(x_periodic),
                jnp.sin(x_periodic),
                jnp.cos(y_periodic),
                jnp.sin(y_periodic),
            ],
            axis=-1,)


class Helmholtz2D(eqx.Module):
    periodic_layer: PeriodicC0Layer
    layers: list

    def __init__(
        self,
        key,
        units: int = 30,
        kmax: int = 10,
        x0: float = -1.0,
        y0: float = -1.0,
        lx: float = 2.0,
        ly: float = 2.0,):
        
        self.periodic_layer = PeriodicC0Layer(
            kmax=kmax,
            x0=x0,
            y0=y0,
            lx=lx,
            ly=ly,)

        keys = jr.split(key, 5)
        in_dim = 4 * kmax

        self.layers = [
            eqx.nn.Linear(in_dim, units, key=keys[0]),
            eqx.nn.Linear(units, units, key=keys[1]),
            eqx.nn.Linear(units, units, key=keys[2]),
            eqx.nn.Linear(units, units, key=keys[3]),
            eqx.nn.Linear(units, 1, key=keys[4]),
        ]

    def __call__(self, x, y):
        xy = jnp.stack([x, y], axis=-1)
        z = self.periodic_layer(xy)

        for layer in self.layers[:-1]:
            z = jax.nn.tanh(layer(z))

        return self.layers[-1](z)


initializer = jax.nn.initializers.glorot_normal()


def init_weight(weight: jax.Array, key: jr.PRNGKey) -> jax.Array:
    out_dim, in_dim = weight.shape
    return initializer(key, shape=(out_dim, in_dim))


def init_linear_layers(model, init_fn, key):
    is_linear = lambda layer: isinstance(layer, eqx.nn.Linear)

    get_weights = lambda m: [
        layer.weight
        for layer in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
        if is_linear(layer)
    ]
    get_biases = lambda m: [
        layer.bias
        for layer in jax.tree_util.tree_leaves(m, is_leaf=is_linear)
        if is_linear(layer)
    ]

    weights = get_weights(model)
    biases = get_biases(model)

    new_weights = [
        init_fn(weight, subkey)
        for weight, subkey in zip(weights, jr.split(key, len(weights)))
    ]
    new_biases = jax.tree_util.tree_map(jnp.zeros_like, biases)

    model = eqx.tree_at(get_weights, model, new_weights)
    model = eqx.tree_at(get_biases, model, new_biases)

    return model


def source_term(x, y):
    exact = jnp.sin(A1 * jnp.pi * x) * jnp.sin(A2 * jnp.pi * y)
    return (K**2 - (A1 * jnp.pi) ** 2 - (A2 * jnp.pi) ** 2) * exact


@eqx.filter_jit
def helmholtz_residual(network, x, y):
    u_fn = lambda x_, y_: network(x_, y_)[0]

    u_x = jax.grad(u_fn, argnums=0)
    u_y = jax.grad(u_fn, argnums=1)

    u_xx = jax.grad(u_x, argnums=0)(x, y)
    u_yy = jax.grad(u_y, argnums=1)(x, y)
    u = u_fn(x, y)

    return (u_xx + u_yy + (K**2) * u - source_term(x, y)) / K


@eqx.filter_jit
def loss_fn(network, xy_r):
    residual = jax.vmap(
        helmholtz_residual,
        in_axes=(None, 0, 0),
    )(network, xy_r[:, 0], xy_r[:, 1])

    return jnp.mean(jnp.square(residual.reshape(-1, 1)))


def run_adam_warmup(network, key: jr.PRNGKey):
    if ADAM_EPOCHS <= 0:
        print("[ADAM] skipped")
        return network, key

    optimizer = optax.radam(learning_rate=ADAM_LR)
    opt_state = optimizer.init(eqx.filter(network, eqx.is_inexact_array))

    @eqx.filter_jit
    def train_step(model, state, xy_r):
        loss_value, grads = eqx.filter_value_and_grad(loss_fn)(model, xy_r)
        updates, new_state = optimizer.update(grads, state, model)
        new_model = eqx.apply_updates(model, updates)
        return new_model, new_state, loss_value

    print("[ADAM] training started")
    t0 = time.time()

    for epoch in range(ADAM_EPOCHS):
        key, data_key = jr.split(key)
        xy_f = generate_inputs(data_key, N_F)

        network, opt_state, loss_value = train_step(network, opt_state, xy_f)

        if epoch == 0 or (epoch + 1) % ADAM_PRINT_EVERY == 0:
            print(f"[ADAM] epoch={epoch + 1} loss={float(loss_value):.7e}")

    print(f"[ADAM] elapsed={time.time() - t0:.2f}s")
    return network, key


class BFGSTrustRegion(optx.AbstractBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBFGSTrustRegion(optx.AbstractSSBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBFGSBacktracking(optx.AbstractSSBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = False
    search: optx.AbstractSearch = optx.BacktrackingArmijo()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBroydenTrustRegion(optx.AbstractSSBroyden):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.LinearTrustRegion()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBFGSStrongWolfe(optx.AbstractSSBFGS):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.BacktrackingStrongWolfe()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


class SSBroydenStrongWolfe(optx.AbstractSSBroyden):
    rtol: float
    atol: float
    norm: Callable = optx.max_norm
    use_inverse: bool = True
    search: optx.AbstractSearch = optx.BacktrackingStrongWolfe()
    descent: optx.AbstractDescent = optx.NewtonDescent()
    verbose: frozenset[str] = frozenset()


def make_solver(method_name: str):
    verbose = frozenset({"loss"})

    solver_classes = {
        "BFGSTrustRegion": BFGSTrustRegion,
        "SSBFGSTrustRegion": SSBFGSTrustRegion,
        "SSBFGSBacktracking": SSBFGSBacktracking,
        "SSBroydenTrustRegion": SSBroydenTrustRegion,
        "SSBFGSStrongWolfe": SSBFGSStrongWolfe,
        "SSBroydenStrongWolfe": SSBroydenStrongWolfe,
    }

    if method_name not in solver_classes:
        raise ValueError(f"Unknown QN_METHOD: {method_name}")

    solver = solver_classes[method_name](rtol=RTOL, atol=ATOL, verbose=verbose)
    return optx.BestSoFarMinimiser(solver)


def run_staged_quasi_newton(network, key: jr.PRNGKey):
    params, static = eqx.partition(network, eqx.is_inexact_array)
    solver = make_solver(QN_METHOD)

    def objective(dynamic_model, static_model, xy_f):
        model = eqx.combine(dynamic_model, static_model)
        return loss_fn(model, xy_f)

    print(f"[{QN_METHOD}] staged training started")
    total_time = 0.0

    for stage_idx, (limit, max_steps) in enumerate(QN_STAGES, start=1):
        key, data_key = jr.split(key)
        xy_f = generate_inputs_growing(data_key, N_F, limit)

        current_model = eqx.combine(params, static)
        start_loss = float(loss_fn(current_model, xy_f))

        print(f"[{QN_METHOD}] stage={stage_idx} limit={limit} max_steps={max_steps}")
        print(f"[{QN_METHOD}] stage={stage_idx} initial_loss={start_loss:.7e}")

        t0 = time.time()

        solution = optx.minimise(
            lambda dyn, stat: objective(dyn, stat, xy_f),
            solver,
            params,
            args=static,
            max_steps=max_steps,
            throw=False,)

        elapsed = time.time() - t0
        total_time += elapsed

        params = solution.value
        trained_model = eqx.combine(params, static)
        final_loss = float(loss_fn(trained_model, xy_f))

        print(f"[{QN_METHOD}] stage={stage_idx} steps={solution.stats['num_steps']}")
        print(f"[{QN_METHOD}] stage={stage_idx} final_loss={final_loss:.7e}")
        print(f"[{QN_METHOD}] stage={stage_idx} elapsed={elapsed:.2f}s")

    network = eqx.combine(params, static)
    print(f"[{QN_METHOD}] total_elapsed={total_time:.2f}s")

    return network, key


def main():
    print("PINN training for the 2D Helmholtz equation")
    print(f"A1={A1}, A2={A2}, K={K}, KMAX={KMAX}")
    print(f"quasi-Newton method: {QN_METHOD}")

    key = jr.PRNGKey(SEED)
    key, init_key = jr.split(key)

    pinn = Helmholtz2D(
        init_key,
        units=UNITS,
        kmax=KMAX,
        x0=X0,
        y0=Y0,
        lx=LX,
        ly=LY,)
    
    pinn = init_linear_layers(pinn, init_weight, init_key)

    pinn, key = run_adam_warmup(pinn, key)
    pinn, key = run_staged_quasi_newton(pinn, key)

    return pinn


if __name__ == "__main__":
    pinn = main()
